In [1]:
import os
import pandas as pd
import numpy as np
import itertools

In [2]:
projectbasepath = "../"
rawdatapath = os.path.join(projectbasepath, "rawdata/")
outdatapath = os.path.join(projectbasepath, "data/")

In [3]:
forecast_date = "2021-01-11"

In [4]:
active = pd.read_csv(os.path.join(rawdatapath, f"forecasts-{forecast_date}/JHMIBedsByDateALL.csv"))
activeicu = pd.read_csv(os.path.join(rawdatapath, f"forecasts-{forecast_date}/JHMIBedsICUByDateALL.csv"))

active_std = pd.read_csv(os.path.join(rawdatapath, f"forecasts-{forecast_date}/JHMIBedsSDByDateALL.csv"))
activeicu_std = pd.read_csv(os.path.join(rawdatapath, f"forecasts-{forecast_date}/JHMIBedsICUSDByDateALL.csv"))

admitted = pd.read_csv(os.path.join(rawdatapath, f"forecasts-{forecast_date}/JHMIArrByDateALL.csv"))
admitted_std = pd.read_csv(os.path.join(rawdatapath, f"forecasts-{forecast_date}/JHMIArrSDByDateALL.csv"))

In [5]:
dropcols = ["Unnamed: 0"]
active = active.drop(columns=dropcols)
activeicu = activeicu.drop(columns=dropcols)
active_std = active_std.drop(columns=dropcols)
activeicu_std = activeicu_std.drop(columns=dropcols)
admitted = admitted.drop(columns=dropcols)
admitted_std = admitted_std.drop(columns=dropcols)

In [6]:
hospitals = ["JHH", "SH", "BMC", "HCGH", "SMH"]
scenarios = {
    "opt":  "optimistic",
    "mod":  "moderate",
    "pess": "pessimistic",
    "cat":  "catastrophic",
}

In [7]:
rename_dict = {ext+" "+h: h+"_"+s for ((ext,s),h) in itertools.product(scenarios.items(), hospitals)}
rename_dict

{'opt JHH': 'JHH_optimistic',
 'opt SH': 'SH_optimistic',
 'opt BMC': 'BMC_optimistic',
 'opt HCGH': 'HCGH_optimistic',
 'opt SMH': 'SMH_optimistic',
 'mod JHH': 'JHH_moderate',
 'mod SH': 'SH_moderate',
 'mod BMC': 'BMC_moderate',
 'mod HCGH': 'HCGH_moderate',
 'mod SMH': 'SMH_moderate',
 'pess JHH': 'JHH_pessimistic',
 'pess SH': 'SH_pessimistic',
 'pess BMC': 'BMC_pessimistic',
 'pess HCGH': 'HCGH_pessimistic',
 'pess SMH': 'SMH_pessimistic',
 'cat JHH': 'JHH_catastrophic',
 'cat SH': 'SH_catastrophic',
 'cat BMC': 'BMC_catastrophic',
 'cat HCGH': 'HCGH_catastrophic',
 'cat SMH': 'SMH_catastrophic'}

In [8]:
rename_dict_std = {ext+" "+h: h+"_"+s+"_std" for ((ext,s),h) in itertools.product(scenarios.items(), hospitals)}

In [9]:
active_nom   = active.rename(columns=rename_dict)
active_std   = active_std.rename(columns=rename_dict_std)
admitted_nom = admitted.rename(columns=rename_dict)
admitted_std = admitted_std.rename(columns=rename_dict_std)
activeicu_nom = activeicu.rename(columns=rename_dict)
activeicu_std = activeicu_std.rename(columns=rename_dict_std)

In [10]:
active_complete = pd.merge(active_nom, active_std, how="outer", on="date")
admitted_complete = pd.merge(admitted_nom, admitted_std, how="outer", on="date")
activeicu_complete = pd.merge(activeicu_nom, activeicu_std, how="outer", on="date")

In [11]:
active_complete = active_complete.fillna(0)
admitted_complete = admitted_complete.fillna(0)
activeicu_complete = activeicu_complete.fillna(0)

In [12]:
num_cols = active_complete.columns.difference(["date"])

In [13]:
active_complete_all = active_complete
active_complete_icu = activeicu_complete
admitted_complete_all = admitted_complete

In [14]:
active_complete_ward = active_complete.copy()
active_complete_ward[num_cols] = active_complete[num_cols].subtract(activeicu_complete[num_cols])

In [15]:
admitted_complete_icu = admitted_complete.copy()
admitted_complete_icu[num_cols] = admitted_complete[num_cols].mul(0.3)

In [16]:
admitted_complete_ward = admitted_complete.copy()
admitted_complete_ward[num_cols] = admitted_complete[num_cols].mul(0.7)

In [17]:
out_date = forecast_date.replace("-", "_")

In [18]:
active_complete_all.to_csv(os.path.join(outdatapath,  f"forecasts-{forecast_date}/jhhs_forecast_active_allbeds_{out_date}.csv"),  date_format="%Y-%m-%d", index=False)
active_complete_icu.to_csv(os.path.join(outdatapath,  f"forecasts-{forecast_date}/jhhs_forecast_active_icu_{out_date}.csv"),  date_format="%Y-%m-%d", index=False)
active_complete_ward.to_csv(os.path.join(outdatapath, f"forecasts-{forecast_date}/jhhs_forecast_active_ward_{out_date}.csv"), date_format="%Y-%m-%d", index=False)

admitted_complete_all.to_csv(os.path.join(outdatapath,  f"forecasts-{forecast_date}/jhhs_forecast_admitted_allbeds_{out_date}.csv"),  date_format="%Y-%m-%d", index=False)
admitted_complete_icu.to_csv(os.path.join(outdatapath,  f"forecasts-{forecast_date}/jhhs_forecast_admitted_icu_{out_date}.csv"),  date_format="%Y-%m-%d", index=False)
admitted_complete_ward.to_csv(os.path.join(outdatapath, f"forecasts-{forecast_date}/jhhs_forecast_admitted_ward_{out_date}.csv"), date_format="%Y-%m-%d", index=False)